<a href="https://colab.research.google.com/github/hamdikhasawneh/AI-sepsis/blob/main/notebooks/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importing file (1) Data Preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
shutil.copy(
    '/content/drive/MyDrive/gp/preprocessing.py',  # ← change this
    '/content/preprocessing.py'
)

'/content/preprocessing.py'

In [ ]:
from preprocessing import (
    run_full_pipeline,
    load_cohort,
    load_vitals_complete,
    load_sofa,
    load_sepsis_labels,
    build_X_array,
    FEATURE_COLS,
)

In [ ]:
from pathlib import Path
OUTPUT_DIR = Path('/content/drive/MyDrive/mimic_iv_processed')
pipeline = run_full_pipeline(force_rebuild=False)
cohort              = pipeline['cohort']
vitals_complete     = pipeline['vitals_complete']
sofa_df             = pipeline['sofa_df']
suspected_infection = pipeline['suspected_infection']
labels_ordered      = pipeline['labels_ordered']
X                   = pipeline['X']
stay_ids_order      = pipeline['stay_ids_order']
label_arrays        = pipeline['label_arrays']

print(f'X shape: {X.shape}')
print(f'Stays  : {len(stay_ids_order):,}')

  ICU Sepsis — Preprocessing Pipeline
[load_cohort] Loaded — shape: (93730, 28)
[load_vitals_complete] Loaded — shape: (2137800, 9)
[build_X_array] X shape: (89075, 24, 7)
[load_sofa] Loaded — shape: (8905908, 12)
[load_suspected_infection] Loaded — shape: (58999, 2)
[load_sepsis_labels] Loaded — shape: (89075, 7)

  Pipeline complete.
  Cohort stays  : 93,730
  X shape       : (89075, 24, 7)
  SOFA rows     : 8,905,908
X shape: (89075, 24, 7)
Stays  : 89,075


## Lab based features beyond SOFA ( Lactate and WBC )
#### Lactate — elevated lactate (>2 mmol/L) is a direct sign of tissue hypoperfusion, which is a core component of septic shock. It's so important that the Sepsis-3 definition uses it to define septic shock specifically.
#### WBC (White Blood Cell count) — abnormally high OR low WBC is a classic SIRS criterion and signals the immune system responding to infection.

In [ ]:
import pandas as pd
d_labitems = pd.read_csv('/content/drive/MyDrive/gp/Cleaned/d_labitems.csv')
print(d_labitems[d_labitems['label'].str.contains('lactate', case=False, na=False)])
print(d_labitems[d_labitems['label'].str.contains('white blood', case=False, na=False)])

      itemid                           label                fluid   category
11     50813                         Lactate                Blood  Blood Gas
41     50843  Lactate Dehydrogenase, Ascites              Ascites  Chemistry
151    50954      Lactate Dehydrogenase (LD)                Blood  Chemistry
242    51054  Lactate Dehydrogenase, Pleural              Pleural  Chemistry
911    51795      Lactate Dehydrogenase, CSF  Cerebrospinal Fluid  Chemistry
1035   51944    Lactate Dehydrogenase, Stool                Stool  Chemistry
1516   52442                         Lactate                Blood  Blood Gas
1615   53154                         Lactate                Blood  Chemistry
     itemid              label  fluid    category
486   51301  White Blood Cells  Blood  Hematology
872   51755  White Blood Cells  Blood   Chemistry
873   51756  White Blood Cells  Blood   Chemistry


In [ ]:
import zipfile

lab_zip_path = Path('/content/drive/MyDrive/gp/Cleaned/chartevents_labevents.zip')
output_file  = OUTPUT_DIR / 'extra_labs_filtered.csv'

EXTRA_LAB_ITEMIDS = [50813, 52442, 51301]  # Lactate, Lactate, WBC

if output_file.exists():
    output_file.unlink()

chunk_size  = 100_000
first_chunk = True
stay_ids    = set(cohort['stay_id'].dropna().unique())

with zipfile.ZipFile(lab_zip_path, 'r') as z:
    with z.open('Cleaned/labevents.csv') as f:
        for i, chunk in enumerate(
            pd.read_csv(
                f,
                chunksize=chunk_size,
                usecols=['subject_id', 'hadm_id', 'charttime', 'itemid', 'valuenum']
            )
        ):
            chunk = chunk[chunk['itemid'].isin(EXTRA_LAB_ITEMIDS)]
            chunk = chunk.merge(
                cohort[['subject_id', 'hadm_id', 'stay_id']],
                on=['subject_id', 'hadm_id'],
                how='inner'
            )
            if not chunk.empty:
                chunk.to_csv(output_file, mode='a', header=first_chunk, index=False)
                first_chunk = False
            print(f'  chunk {i+1} done', end='\r')

print('\nextra_labs_filtered.csv saved successfully')

  chunk 1584 done
extra_labs_filtered.csv saved successfully


In [ ]:
extra_labs = pd.read_csv(OUTPUT_DIR / 'extra_labs_filtered.csv')
extra_labs['charttime'] = pd.to_datetime(extra_labs['charttime'])
print('Shape:', extra_labs.shape)
print('\nItemid counts:')
print(extra_labs['itemid'].value_counts())
print('\nMissing values:')
print(extra_labs.isnull().sum())
print('\nSample:')
extra_labs.head()

Shape: (1822267, 6)

Itemid counts:
itemid
51301    1329151
50813     493116
Name: count, dtype: int64

Missing values:
subject_id       0
hadm_id          0
itemid           0
charttime        0
valuenum      4214
stay_id          0
dtype: int64

Sample:


,subject_id,hadm_id,itemid,charttime,valuenum,stay_id
0,10000032,29079034.0,51301,2180-07-24 06:35:00,4.1,39553978
1,10000032,29079034.0,51301,2180-07-25 04:45:00,4.8,39553978
2,10000690,25860671.0,51301,2150-11-03 02:56:00,7.5,37081114
3,10000690,25860671.0,51301,2150-11-04 03:03:00,6.1,37081114
4,10000690,25860671.0,51301,2150-11-05 05:36:00,6.2,37081114


Cleaning File

In [ ]:
# Drop missing values
extra_labs = extra_labs.dropna(subset=['valuenum'])
print('After dropping nulls:', extra_labs.shape)

# Map itemids to names
extra_lab_itemid_to_name = {
    50813: 'lactate',
    52442: 'lactate',
    51301: 'wbc',
}
extra_labs['lab_name'] = extra_labs['itemid'].map(extra_lab_itemid_to_name)

# Floor to hourly bins
extra_labs['charttime_hour'] = extra_labs['charttime'].dt.floor('h')

# Aggregate to hourly mean
extra_labs_hourly = (
    extra_labs
    .groupby(['stay_id', 'charttime_hour', 'lab_name'])['valuenum']
    .mean()
    .reset_index()
)

# Pivot to wide format
extra_labs_wide = extra_labs_hourly.pivot_table(
    index=['stay_id', 'charttime_hour'],
    columns='lab_name',
    values='valuenum'
).reset_index()
extra_labs_wide.columns.name = None

print('Wide format shape:', extra_labs_wide.shape)
extra_labs_wide.head()

After dropping nulls: (1818053, 6)
Wide format shape: (1640451, 4)


,stay_id,charttime_hour,lactate,wbc
0,30000153,2174-09-29 13:00:00,1.3,NaN
1,30000153,2174-09-29 14:00:00,2.1,NaN
2,30000153,2174-09-29 15:00:00,NaN,17.0
3,30000153,2174-09-30 03:00:00,NaN,15.2
4,30000153,2174-10-01 05:00:00,NaN,11.9


In [ ]:
print(cohort.columns.tolist())
print(cohort[['stay_id', 'intime']].head())

['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod', 'icu_los_hours']
    stay_id              intime
0  39553978 2180-07-23 14:00:00
1  37081114 2150-11-02 19:37:00
2  39765666 2189-06-27 08:42:00
3  37067082 2157-11-20 19:18:02
4  34592300 2157-12-19 15:42:24


In [ ]:
print(extra_labs_wide.columns.tolist())

['stay_id', 'charttime_hour', 'lactate', 'wbc']


In [ ]:
# Rebuild extra_labs_wide cleanly from extra_labs_hourly
extra_labs_wide = extra_labs_hourly.pivot_table(
    index=['stay_id', 'charttime_hour'],
    columns='lab_name',
    values='valuenum'
).reset_index()
extra_labs_wide.columns.name = None

# Now merge intime once
extra_labs_wide['charttime_hour'] = pd.to_datetime(extra_labs_wide['charttime_hour'])
extra_labs_wide = extra_labs_wide.merge(
    cohort[['stay_id', 'intime']],
    on='stay_id',
    how='left'
)
extra_labs_wide['intime'] = pd.to_datetime(extra_labs_wide['intime'])

# Hours since admission
extra_labs_wide['hours_since_admit'] = (
    (extra_labs_wide['charttime_hour'] - extra_labs_wide['intime'])
    .dt.total_seconds() / 3600
)

# Keep first 24 hours only
extra_labs_24h = extra_labs_wide[
    (extra_labs_wide['hours_since_admit'] >= 0) &
    (extra_labs_wide['hours_since_admit'] <  24)
].copy()

extra_labs_24h['hour'] = extra_labs_24h['hours_since_admit'].astype(int)

print('Shape after 24h filter:', extra_labs_24h.shape)
print('Stays covered:', extra_labs_24h['stay_id'].nunique())
extra_labs_24h[['stay_id', 'hour', 'lactate', 'wbc']].head(10)

Shape after 24h filter: (264931, 7)
Stays covered: 90325


,stay_id,hour,lactate,wbc
0,30000153,0,1.3,NaN
1,30000153,1,2.1,NaN
2,30000153,2,NaN,17.0
3,30000153,14,NaN,15.2
15,30000213,2,0.9,NaN
16,30000213,22,NaN,5.8
21,30000484,3,2.0,NaN
22,30000484,10,NaN,24.2
23,30000484,11,1.6,NaN
34,30000646,0,NaN,8.5


In [ ]:
import numpy as np
def compute_lab_features(extra_labs_24h: pd.DataFrame) -> pd.DataFrame:
    """
    For each stay compute summary statistics for lactate and WBC
    over the first 24 hours.

    NOTE: slope is computed only on *observed* (non-imputed) values,
    using the original charttime_hour as the x-axis.  This avoids
    reporting trend artefacts from forward-fill imputation.
    Patients with fewer than 2 real measurements get slope = NaN.
    """
    records = []

    for stay_id, group in extra_labs_24h.groupby('stay_id'):
        row = {'stay_id': stay_id}

        for lab in ['lactate', 'wbc']:
            # Use only actually-observed rows (not forward-filled ones).
            # extra_labs_24h must carry an 'observed' flag OR we rely on
            # the raw merge before ffill; here we use the pre-ffill counts
            # stored in *_count to guard slope computation.
            vals = group[lab].dropna().values

            if len(vals) == 0:
                row[f'{lab}_mean']    = np.nan
                row[f'{lab}_max']     = np.nan
                row[f'{lab}_min']     = np.nan
                row[f'{lab}_first']   = np.nan
                row[f'{lab}_last']    = np.nan
                row[f'{lab}_count']   = 0
                row[f'{lab}_missing'] = 1
                row[f'{lab}_slope']   = np.nan   # not enough data
            else:
                row[f'{lab}_mean']    = np.nanmean(vals)
                row[f'{lab}_max']     = np.nanmax(vals)
                row[f'{lab}_min']     = np.nanmin(vals)
                row[f'{lab}_first']   = vals[0]
                row[f'{lab}_last']    = vals[-1]
                row[f'{lab}_count']   = len(vals)
                row[f'{lab}_missing'] = 0

                # ── Slope fix: only on observed rows, using real hours ──
                # 'observed_{lab}' flag is set to 1 only for non-imputed rows.
                # If the flag column is absent (older pipeline), fall back to
                # all non-NaN rows — still safer than slope over a filled grid.
                obs_flag = f'observed_{lab}'
                if obs_flag in group.columns:
                    obs_rows = group[group[obs_flag] == 1]
                else:
                    obs_rows = group[group[lab].notna()]

                obs_vals  = obs_rows[lab].values.astype(float)
                obs_hours = obs_rows['hour'].values.astype(float)

                if len(obs_vals) >= 2:
                    from scipy.stats import linregress
                    slope, *_ = linregress(obs_hours, obs_vals)
                    row[f'{lab}_slope'] = slope
                else:
                    row[f'{lab}_slope'] = np.nan   # unreliable with <2 points

        # Clinically important thresholds
        lactate_vals = group['lactate'].dropna().values
        wbc_vals     = group['wbc'].dropna().values

        row['lactate_elevated'] = int(any(lactate_vals > 2.0))
        row['lactate_critical'] = int(any(lactate_vals > 4.0))
        row['wbc_high']         = int(any(wbc_vals > 12.0))
        row['wbc_low']          = int(any(wbc_vals < 4.0))
        row['wbc_abnormal']     = int(row['wbc_high'] or row['wbc_low'])

        records.append(row)

    return pd.DataFrame(records)


lab_features = compute_lab_features(extra_labs_24h)
print('Lab features shape:', lab_features.shape)
print('\nMissing values:')
print(lab_features.isnull().sum())
lab_features.head()


Lab features shape: (90325, 20)

Missing values:
stay_id                 0
lactate_mean        41307
lactate_max         41307
lactate_min         41307
lactate_first       41307
lactate_last        41307
lactate_count           0
lactate_missing         0
wbc_mean              897
wbc_max               897
wbc_min               897
wbc_first             897
wbc_last              897
wbc_count               0
wbc_missing             0
lactate_elevated        0
lactate_critical        0
wbc_high                0
wbc_low                 0
wbc_abnormal            0
dtype: int64


,stay_id,lactate_mean,lactate_max,lactate_min,lactate_first,lactate_last,lactate_count,lactate_missing,wbc_mean,wbc_max,wbc_min,wbc_first,wbc_last,wbc_count,wbc_missing,lactate_elevated,lactate_critical,wbc_high,wbc_low,wbc_abnormal
0,30000153,1.7,2.1,1.3,1.3,2.1,2,0,16.1,17.0,15.2,17.0,15.2,2,0,1,0,1,0,1
1,30000213,0.9,0.9,0.9,0.9,0.9,1,0,5.8,5.8,5.8,5.8,5.8,1,0,0,0,0,0,0
2,30000484,1.8,2.0,1.6,2.0,1.6,2,0,24.2,24.2,24.2,24.2,24.2,1,0,0,0,1,0,1
3,30000646,1.6,1.6,1.6,1.6,1.6,2,0,9.0,10.6,7.9,8.5,7.9,3,0,0,0,0,0,0
4,30000831,1.4,1.4,1.4,1.4,1.4,2,0,14.2,14.2,14.2,14.2,14.2,1,0,0,0,1,0,1


## Running remaining feature engineering sections

In [ ]:
from scipy.stats import linregress

# Section 2 - Temporal vital sign features
def compute_temporal_features(vitals_complete, feature_cols):
    """
    Compute per-stay summary stats and trends over the first 24h.

    Slope fix: linregress is run ONLY on hours where the raw sensor
    actually recorded a value.  vitals_complete must include an
    'observed_{col}' boolean/int column for each vital (set to 1 before
    forward-fill, 0 on imputed rows).  If those flags are absent, the
    function falls back to all non-NaN rows — still better than running
    regression over a forward-filled series.
    """
    records = []
    for stay_id, group in vitals_complete.groupby('stay_id'):
        group = group.sort_values('hour')
        row = {'stay_id': stay_id}
        hours = group['hour'].values.astype(float)

        for col in feature_cols:
            vals = group[col].values.astype(float)
            row[f'{col}_mean'] = np.nanmean(vals)
            row[f'{col}_std']  = np.nanstd(vals)
            row[f'{col}_min']  = np.nanmin(vals)
            row[f'{col}_max']  = np.nanmax(vals)
            row[f'{col}_last'] = vals[-1] if not np.isnan(vals[-1]) else np.nanmean(vals)

            # ── Slope: use only observed (pre-imputation) rows ──────────
            obs_flag = f'observed_{col}'
            if obs_flag in group.columns:
                obs_mask = group[obs_flag].values == 1
            else:
                obs_mask = ~np.isnan(vals)      # fallback

            obs_hours = hours[obs_mask]
            obs_vals  = vals[obs_mask]

            if len(obs_vals) >= 2:
                slope, _, _, _, _ = linregress(obs_hours, obs_vals)
                row[f'{col}_slope'] = slope
            else:
                row[f'{col}_slope'] = np.nan    # not enough real readings

        records.append(row)
    return pd.DataFrame(records)

print('Computing temporal features — this will take a few minutes...')
temporal_features = compute_temporal_features(vitals_complete, FEATURE_COLS)
print(f'Temporal features shape: {temporal_features.shape}')


Computing temporal features — this will take a few minutes...


/tmp/ipykernel_8495/3010317070.py:10: RuntimeWarning: Mean of empty slice
  row[f'{col}_mean']  = np.nanmean(vals)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_8495/3010317070.py:12: RuntimeWarning: All-NaN slice encountered
  row[f'{col}_min']   = np.nanmin(vals)
/tmp/ipykernel_8495/3010317070.py:13: RuntimeWarning: All-NaN slice encountered
  row[f'{col}_max']   = np.nanmax(vals)
/tmp/ipykernel_8495/3010317070.py:14: RuntimeWarning: Mean of empty slice
  row[f'{col}_last']  = vals[-1] if not np.isnan(vals[-1]) else np.nanmean(vals)


Temporal features shape: (89075, 43)


In [ ]:
# SOFA feature section
# Section 3 - SOFA features
def compute_sofa_features(sofa_df, cohort):
    sofa = sofa_df.copy()
    sofa['charttime_hour'] = pd.to_datetime(sofa['charttime_hour'])
    sofa = sofa.merge(cohort[['stay_id', 'intime']], on='stay_id', how='left')
    sofa['intime'] = pd.to_datetime(sofa['intime'])
    sofa['hours_since_admit'] = (
        (sofa['charttime_hour'] - sofa['intime']).dt.total_seconds() / 3600
    )
    sofa_24h = sofa[
        (sofa['hours_since_admit'] >= 0) &
        (sofa['hours_since_admit'] <  24)
    ].copy()
    sofa_24h['hour'] = sofa_24h['hours_since_admit'].astype(int)

    records = []
    for stay_id, group in sofa_24h.groupby('stay_id'):
        group = group.sort_values('hour')
        vals  = group['sofa_total'].values.astype(float)
        h     = group['hour'].values
        mask  = ~np.isnan(vals)

        slope = np.nan
        if mask.sum() >= 2:
            slope, *_ = linregress(h[mask], vals[mask])

        records.append({
            'stay_id':        stay_id,
            'sofa_max_24h':   np.nanmax(vals)  if mask.any() else np.nan,
            'sofa_mean_24h':  np.nanmean(vals) if mask.any() else np.nan,
            'sofa_last_24h':  vals[-1] if not np.isnan(vals[-1]) else np.nanmean(vals),
            'sofa_slope_24h': slope,
            'sofa_hours_ge2': int((vals >= 2).sum()),
        })

    return pd.DataFrame(records)
print('Computing SOFA features...')
sofa_features = compute_sofa_features(sofa_df, cohort)
print(f'SOFA features shape: {sofa_features.shape}')

Computing SOFA features...


/tmp/ipykernel_8495/3059103048.py:32: RuntimeWarning: Mean of empty slice
  'sofa_last_24h':  vals[-1] if not np.isnan(vals[-1]) else np.nanmean(vals),


SOFA features shape: (93714, 6)


# Static Patients Features

In [ ]:
# Section 4 - Static patient features
def compute_static_features(cohort):
    """
    Build static features from cohort demographics.

    FIX (issue #3): 'icu_los_hours' is removed.
    ICU LOS is not knowable at prediction time (first 24h window) and
    passing it to the model constitutes a direct data leak.
    """
    static = cohort[[
        'stay_id', 'anchor_age', 'gender',
        'admission_type', 'admission_location',
        # 'icu_los_hours'  ← REMOVED: future leakage
    ]].copy()

    # Binary gender encoding
    static['gender_male'] = (static['gender'].str.upper() == 'M').astype(int)

    # One-hot encode admission type
    admission_dummies = pd.get_dummies(
        static['admission_type'].str.lower().str.replace(' ', '_'),
        prefix='adm_type'
    ).astype(int)
    static = pd.concat([static, admission_dummies], axis=1)

    # Drop original string columns
    static = static.drop(columns=['gender', 'admission_type', 'admission_location'])

    return static

static_features = compute_static_features(cohort)
print(f'Static features shape: {static_features.shape}')
print('\nColumns:', static_features.columns.tolist())
static_features.head()


Static features shape: (93730, 13)

Columns: ['stay_id', 'anchor_age', 'icu_los_hours', 'gender_male', 'adm_type_ambulatory_observation', 'adm_type_direct_emer.', 'adm_type_direct_observation', 'adm_type_elective', 'adm_type_eu_observation', 'adm_type_ew_emer.', 'adm_type_observation_admit', 'adm_type_surgical_same_day_admission', 'adm_type_urgent']


,stay_id,anchor_age,icu_los_hours,gender_male,adm_type_ambulatory_observation,adm_type_direct_emer.,adm_type_direct_observation,adm_type_elective,adm_type_eu_observation,adm_type_ew_emer.,adm_type_observation_admit,adm_type_surgical_same_day_admission,adm_type_urgent
0,39553978,52,9.846389,0,0,0,0,0,0,1,0,0,0
1,37081114,86,93.438056,0,0,0,0,0,0,1,0,0,0
2,39765666,73,11.940833,0,0,0,0,0,0,1,0,0,0
3,37067082,55,26.832778,0,0,0,0,0,0,1,0,0,0
4,34592300,55,22.754722,0,0,1,0,0,0,0,0,0,0


In [ ]:
# Section 5 - Missingness indicators
def compute_missingness_features(vitals_complete, feature_cols):
    records = []
    for stay_id, group in vitals_complete.groupby('stay_id'):
        row = {'stay_id': stay_id}
        for col in feature_cols:
            n_missing = group[col].isna().sum()
            row[f'{col}_missing_frac'] = n_missing / len(group)
        records.append(row)
    return pd.DataFrame(records)

print('Computing missingness features...')
missingness_features = compute_missingness_features(vitals_complete, FEATURE_COLS)
print(f'Missingness features shape: {missingness_features.shape}')
missingness_features.head()

Computing missingness features...
Missingness features shape: (89075, 8)


,stay_id,abp_dia_missing_frac,abp_mean_missing_frac,abp_sys_missing_frac,heart_rate_missing_frac,resp_rate_missing_frac,spo2_missing_frac,temp_c_missing_frac
0,30000153,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,30000213,1.0,1.0,1.0,0.0,0.0,0.0,1.0
2,30000484,1.0,1.0,1.0,0.0,0.0,0.0,1.0
3,30000646,1.0,1.0,1.0,0.0,0.0,0.0,1.0
4,30000831,1.0,1.0,1.0,0.0,0.0,0.0,1.0


## Combining Final Features into one table

In [ ]:
# Section 6 - Combine all features
# FIX (issue #2): label columns (y_*, eligible_*) are NOT merged into
# all_features here.  They are saved in a separate file so File 3 cannot
# accidentally include them as model inputs.

LABEL_COLS_IN_LABELS_ORDERED = ['stay_id', 'eligible_6h', 'eligible_12h',
                                  'eligible_24h', 'y_6h', 'y_12h', 'y_24h']
# Separate labels from the pipeline output
labels_df = labels_ordered[LABEL_COLS_IN_LABELS_ORDERED].copy()

# Feature-only merge (no labels)
all_features = (
    pd.DataFrame({'stay_id': stay_ids_order})
    .merge(temporal_features,    on='stay_id', how='left')
    .merge(sofa_features,        on='stay_id', how='left')
    .merge(static_features,      on='stay_id', how='left')
    .merge(missingness_features, on='stay_id', how='left')
    .merge(lab_features,         on='stay_id', how='left')
    # labels_df intentionally excluded
)

print(f'Feature table shape (no labels): {all_features.shape}')
print(f'\nTotal feature columns: {all_features.shape[1] - 1}')  # -1 for stay_id
print(f'\nMissing values (top 15):')
print(all_features.isnull().sum().sort_values(ascending=False).head(15))


Final feature table shape: (89075, 92)

Total features: 92

Missing values (top 15):
temp_c_slope     81711
temp_c_last      81711
temp_c_min       81711
temp_c_max       81711
temp_c_mean      81711
temp_c_std       81711
abp_sys_last     57499
abp_sys_slope    57499
abp_sys_max      57499
abp_sys_min      57499
abp_sys_std      57499
abp_sys_mean     57499
abp_dia_mean     57493
abp_dia_std      57493
abp_dia_slope    57493
dtype: int64


## Handeling missingness

In [ ]:
# Carry forward last known lactate and WBC within each stay
extra_labs_24h = extra_labs_24h.sort_values(['stay_id', 'hour'])

extra_labs_24h['lactate'] = (
    extra_labs_24h.groupby('stay_id')['lactate']
    .transform(lambda x: x.ffill())
)

extra_labs_24h['wbc'] = (
    extra_labs_24h.groupby('stay_id')['wbc']
    .transform(lambda x: x.ffill())
)

print('After forward fill:')
print(extra_labs_24h[['stay_id', 'hour', 'lactate', 'wbc']].head(15))
print('\nRemaining NaN:')
print(extra_labs_24h[['lactate', 'wbc']].isnull().sum())

After forward fill:
     stay_id  hour  lactate   wbc
0   30000153     0      1.3   NaN
1   30000153     1      2.1   NaN
2   30000153     2      2.1  17.0
3   30000153    14      2.1  15.2
15  30000213     2      0.9   NaN
16  30000213    22      0.9   5.8
21  30000484     3      2.0   NaN
22  30000484    10      2.0  24.2
23  30000484    11      1.6  24.2
34  30000646     0      NaN   8.5
35  30000646     4      1.6  10.6
36  30000646     8      1.6  10.6
37  30000646    17      1.6   7.9
47  30000831     7      1.4  14.2
48  30000831    14      1.4  14.2

Remaining NaN:
lactate    77616
wbc        25587
dtype: int64


#### forward fill for patients that measures have been taken within 24h and population median fill for patients that have been note taken

In [ ]:
# Recompute lab features with forward-filled data
print('Recomputing lab features with forward-filled values...')
lab_features = compute_lab_features(extra_labs_24h)
print(f'Lab features shape: {lab_features.shape}')
print('\nMissing values:')
print(lab_features.isnull().sum())

Recomputing lab features with forward-filled values...
Lab features shape: (90325, 20)

Missing values:
stay_id                 0
lactate_mean        41307
lactate_max         41307
lactate_min         41307
lactate_first       41307
lactate_last        41307
lactate_count           0
lactate_missing         0
wbc_mean              897
wbc_max               897
wbc_min               897
wbc_first             897
wbc_last              897
wbc_count               0
wbc_missing             0
lactate_elevated        0
lactate_critical        0
wbc_high                0
wbc_low                 0
wbc_abnormal            0
dtype: int64


### Build the final all_features table with the updated lab features and then median fill

In [ ]:
# Section 6 - Rebuild combined feature table with updated lab features
# FIX (issue #2): labels kept separate — see labels_df defined above.
all_features = (
    pd.DataFrame({'stay_id': stay_ids_order})
    .merge(temporal_features,    on='stay_id', how='left')
    .merge(sofa_features,        on='stay_id', how='left')
    .merge(static_features,      on='stay_id', how='left')
    .merge(missingness_features, on='stay_id', how='left')
    .merge(lab_features,         on='stay_id', how='left')
    # labels_df excluded — saved separately
)

print(f'Combined feature table shape: {all_features.shape}')

# Section 7 - Fill remaining NaN with population median
# (applies only to feature columns — labels are in a separate dataframe)
feature_cols_all = [c for c in all_features.columns if c != 'stay_id']

# Store medians so we can reuse them at inference time
medians = all_features[feature_cols_all].median()
all_features[feature_cols_all] = all_features[feature_cols_all].fillna(medians)

print('\nMissing values after median fill:')
print(all_features.isnull().sum().sum())
print(f'\nFinal shape: {all_features.shape}')
print(f'Total features (excl. stay_id): {len(feature_cols_all)}')


Combined feature table shape: (89075, 92)

Missing values after median fill:
0

Final shape: (89075, 92)
Total features: 85


In [ ]:
# Section 8 - Save all outputs

# ── Feature table (NO labels) ──────────────────────────────────────────
all_features.to_csv(OUTPUT_DIR / 'engineered_features.csv', index=False)
print(f'Saved engineered_features.csv — shape: {all_features.shape}')

# ── Labels saved separately (FIX issue #2) ────────────────────────────
# File 3 should load this independently and join on stay_id only when
# needed.  This prevents accidental inclusion of label columns as inputs.
labels_df.to_csv(OUTPUT_DIR / 'labels.csv', index=False)
print(f'Saved labels.csv — shape: {labels_df.shape}')
print(f'  Columns: {labels_df.columns.tolist()}')

# ── Medians for inference-time imputation ──────────────────────────────
medians.to_csv(OUTPUT_DIR / 'feature_medians.csv', header=True)
print('Saved feature_medians.csv')

# ── LSTM 3-D array ─────────────────────────────────────────────────────
np.save(OUTPUT_DIR / 'X_vitals.npy', X)
print(f'Saved X_vitals.npy — shape: {X.shape}')

# ── Feature column names ───────────────────────────────────────────────
feature_cols_all = [c for c in all_features.columns if c != 'stay_id']
with open(OUTPUT_DIR / 'feature_names.txt', 'w') as f:
    f.write('\n'.join(feature_cols_all))
print(f'Saved feature_names.txt — {len(feature_cols_all)} features')

# ── Sanity check: confirm no label columns leaked into features ────────
leaked = [c for c in ['y_6h','y_12h','y_24h','eligible_6h','eligible_12h','eligible_24h']
          if c in all_features.columns]
assert len(leaked) == 0, f'LEAKAGE: label columns in feature table: {leaked}'
print('\nLeakage check passed — no label columns in engineered_features.csv')

# Section 9 - Label summary
print('\n--- Label Summary ---')
for horizon in ['6h', '12h', '24h']:
    y    = label_arrays[f'y_{horizon}']
    elig = label_arrays[f'eligible_{horizon}']
    print(f'Horizon {horizon:>3s} | eligible: {elig.sum():,} '
          f'| sepsis positive: {y[elig].sum():,} '
          f'| positive rate: {y[elig].mean():.3f}')


Saved engineered_features.csv — shape: (89075, 92)
Saved feature_medians.csv
Saved X_vitals.npy — shape: (89075, 24, 7)
Saved feature_names.txt — 85 features

--- Label Summary ---
Horizon  6h | eligible: 64,815 | sepsis positive: 297 | positive rate: 0.005
Horizon 12h | eligible: 58,021 | sepsis positive: 453 | positive rate: 0.008
Horizon 24h | eligible: 46,157 | sepsis positive: 662 | positive rate: 0.014


## Gap Features: Urine Output, Vasopressor Flag, Ventilation Status, Blood Culture

These four features appear in EPIC Sepsis Model and Sepsis Watch but were absent.
They are extracted here and merged into `all_features` before saving.
Run this block after Section 8 above re-saves `engineered_features.csv`.


In [ ]:
# ── FIX (issue #4): Missing features from published models ───────────────────
#
#  1. Urine output (outputevents)      — organ dysfunction signal
#  2. Vasopressor flag (inputevents)   — already extracted in preprocessing
#     if vasopressors_filtered.csv is present; binary flag here
#  3. Ventilation status (chartevents) — procedureevents / ventilation table
#  4. Blood culture result             — microbiologyevents (positive flag)
#
# All four are joined to all_features and the combined CSV is re-saved.
# ─────────────────────────────────────────────────────────────────────────────

import zipfile, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

DATA_DIR = Path('/content/drive/MyDrive/gp/Cleaned')

# ── 1. URINE OUTPUT ─────────────────────────────────────────────────────────
uo_output = OUTPUT_DIR / 'urine_output_filtered.csv'
URINE_ITEMIDS = [
    226559, 226560, 226561, 226584, 226563, 226564, 226565, 226567,
    226557, 226558, 227488, 227489,  # Foley, straight cath, etc.
]
stay_ids = set(cohort['stay_id'].dropna().unique())

if not uo_output.exists():
    first_chunk = True
    with open(DATA_DIR / 'outputevents.csv') as f:
        for i, chunk in enumerate(pd.read_csv(
            f, chunksize=100_000,
            usecols=['stay_id', 'charttime', 'itemid', 'value']
        )):
            chunk = chunk[
                chunk['stay_id'].isin(stay_ids) &
                chunk['itemid'].isin(URINE_ITEMIDS)
            ]
            if not chunk.empty:
                chunk.to_csv(uo_output, mode='a', header=first_chunk, index=False)
                first_chunk = False
        print(f'UO chunk {i+1}', end='\r')
    print('\nurine_output_filtered.csv saved')
else:
    print('urine_output_filtered.csv already exists, skipping extraction')

uo_df = pd.read_csv(uo_output)
uo_df['charttime'] = pd.to_datetime(uo_df['charttime'])
uo_df = uo_df.merge(cohort[['stay_id', 'intime']], on='stay_id', how='left')
uo_df['intime'] = pd.to_datetime(uo_df['intime'])
uo_df['hours_since_admit'] = (
    (uo_df['charttime'] - uo_df['intime']).dt.total_seconds() / 3600
)
uo_24h = uo_df[
    (uo_df['hours_since_admit'] >= 0) &
    (uo_df['hours_since_admit'] <  24)
].copy()
uo_24h['value'] = pd.to_numeric(uo_24h['value'], errors='coerce')

uo_features = (
    uo_24h.groupby('stay_id')['value']
    .agg(
        uo_total_24h='sum',
        uo_mean_hourly=lambda x: x.sum() / 24,
        uo_min_hourly='min',
        uo_count='count'
    )
    .reset_index()
)
# Clinical flag: oliguria = total UO < 500 mL in 24h
uo_features['oliguria_flag'] = (uo_features['uo_total_24h'] < 500).astype(int)
print(f'UO features shape: {uo_features.shape}')

# ── 2. VASOPRESSOR FLAG ──────────────────────────────────────────────────────
# vasopressors_filtered.csv is produced by the preprocessing pipeline (Block 6)
vaso_path = OUTPUT_DIR / 'vasopressors_filtered.csv'
if vaso_path.exists():
    vaso_df = pd.read_csv(vaso_path)
    vaso_df['starttime'] = pd.to_datetime(vaso_df['starttime'])
    vaso_df = vaso_df.merge(cohort[['stay_id', 'intime']], on='stay_id', how='left')
    vaso_df['intime'] = pd.to_datetime(vaso_df['intime'])
    vaso_df['hours_since_admit'] = (
        (vaso_df['starttime'] - vaso_df['intime']).dt.total_seconds() / 3600
    )
    vaso_24h = vaso_df[
        (vaso_df['hours_since_admit'] >= 0) &
        (vaso_df['hours_since_admit'] <  24)
    ]
    vaso_features = (
        vaso_24h.groupby('stay_id')
        .size()
        .reset_index(name='vaso_events_24h')
    )
    vaso_features['vasopressor_flag'] = 1
    # Stays with no vasopressor records get flag = 0
    vaso_features = cohort[['stay_id']].merge(
        vaso_features[['stay_id', 'vaso_events_24h', 'vasopressor_flag']],
        on='stay_id', how='left'
    )
    vaso_features['vasopressor_flag']   = vaso_features['vasopressor_flag'].fillna(0).astype(int)
    vaso_features['vaso_events_24h']    = vaso_features['vaso_events_24h'].fillna(0).astype(int)
    print(f'Vasopressor features shape: {vaso_features.shape}')
    print(f'  Patients on vasopressors in first 24h: {vaso_features["vasopressor_flag"].sum():,}')
else:
    print('WARNING: vasopressors_filtered.csv not found — vasopressor features skipped.')
    print('Run preprocessing Block 6 to generate it.')
    vaso_features = cohort[['stay_id']].copy()
    vaso_features['vasopressor_flag'] = np.nan
    vaso_features['vaso_events_24h']  = np.nan

# ── 3. VENTILATION STATUS ────────────────────────────────────────────────────
# MIMIC-IV ventilation table (if present) or procedureevents fallback
vent_path = DATA_DIR / 'ventilation.csv'
vent_proc_path = DATA_DIR / 'procedureevents.csv'

VENT_PROC_ITEMIDS = [225792, 225794]  # Invasive/non-invasive vent in procedureevents

if vent_path.exists():
    vent_raw = pd.read_csv(vent_path, usecols=['stay_id', 'starttime', 'ventilation_status'])
    vent_raw['starttime'] = pd.to_datetime(vent_raw['starttime'])
    vent_raw = vent_raw.merge(cohort[['stay_id', 'intime']], on='stay_id', how='left')
    vent_raw['intime'] = pd.to_datetime(vent_raw['intime'])
    vent_raw['hours_since_admit'] = (
        (vent_raw['starttime'] - vent_raw['intime']).dt.total_seconds() / 3600
    )
    vent_24h = vent_raw[
        (vent_raw['hours_since_admit'] >= 0) &
        (vent_raw['hours_since_admit'] <  24) &
        (vent_raw['ventilation_status'] == 'InvasiveVent')
    ]
elif vent_proc_path.exists():
    vent_raw = pd.read_csv(vent_proc_path,
                           usecols=['stay_id', 'starttime', 'itemid'])
    vent_raw = vent_raw[vent_raw['itemid'].isin(VENT_PROC_ITEMIDS)]
    vent_raw['starttime'] = pd.to_datetime(vent_raw['starttime'])
    vent_raw = vent_raw.merge(cohort[['stay_id', 'intime']], on='stay_id', how='left')
    vent_raw['intime'] = pd.to_datetime(vent_raw['intime'])
    vent_raw['hours_since_admit'] = (
        (vent_raw['starttime'] - vent_raw['intime']).dt.total_seconds() / 3600
    )
    vent_24h = vent_raw[
        (vent_raw['hours_since_admit'] >= 0) &
        (vent_raw['hours_since_admit'] <  24)
    ]
else:
    print('WARNING: Neither ventilation.csv nor procedureevents.csv found.')
    vent_24h = pd.DataFrame(columns=['stay_id'])

ventilated_stays = set(vent_24h['stay_id'].unique()) if len(vent_24h) > 0 else set()
vent_features = cohort[['stay_id']].copy()
vent_features['ventilated_flag'] = vent_features['stay_id'].isin(ventilated_stays).astype(int)
print(f'Ventilation features shape: {vent_features.shape}')
print(f'  Ventilated patients in first 24h: {vent_features["ventilated_flag"].sum():,}')

# ── 4. BLOOD CULTURE RESULT ──────────────────────────────────────────────────
micro_path = DATA_DIR / 'microbiologyevents.csv'
if micro_path.exists():
    micro_df = pd.read_csv(micro_path,
                            usecols=['subject_id', 'hadm_id', 'charttime',
                                     'spec_type_desc', 'org_name'])
    micro_df = micro_df[
        micro_df['spec_type_desc'].str.contains('BLOOD', na=False, case=False)
    ]
    micro_df = micro_df.merge(
        cohort[['subject_id', 'hadm_id', 'stay_id', 'intime']],
        on=['subject_id', 'hadm_id'], how='inner'
    )
    micro_df['charttime'] = pd.to_datetime(micro_df['charttime'])
    micro_df['intime']    = pd.to_datetime(micro_df['intime'])
    micro_df['hours_since_admit'] = (
        (micro_df['charttime'] - micro_df['intime']).dt.total_seconds() / 3600
    )
    # Use a 72h window for cultures (results often arrive 24–48h after draw)
    micro_24_72 = micro_df[
        (micro_df['hours_since_admit'] >= 0) &
        (micro_df['hours_since_admit'] <= 72)
    ]
    # Positive = any organism grown
    micro_24_72 = micro_24_72.copy()
    micro_24_72['culture_positive'] = micro_24_72['org_name'].notna().astype(int)

    culture_features = (
        micro_24_72.groupby('stay_id')
        .agg(
            blood_culture_drawn=('culture_positive', 'count'),
            blood_culture_positive=('culture_positive', 'max')
        )
        .reset_index()
    )
    culture_features = cohort[['stay_id']].merge(culture_features, on='stay_id', how='left')
    culture_features['blood_culture_drawn']    = culture_features['blood_culture_drawn'].fillna(0).astype(int)
    culture_features['blood_culture_positive'] = culture_features['blood_culture_positive'].fillna(0).astype(int)
    print(f'Culture features shape: {culture_features.shape}')
    print(f'  Blood culture drawn: {culture_features["blood_culture_drawn"].gt(0).sum():,}')
    print(f'  Blood culture positive: {culture_features["blood_culture_positive"].sum():,}')
else:
    print('WARNING: microbiologyevents.csv not found — culture features skipped.')
    culture_features = cohort[['stay_id']].copy()
    culture_features['blood_culture_drawn']    = np.nan
    culture_features['blood_culture_positive'] = np.nan

# ── Merge all gap features into all_features and re-save ─────────────────────
all_features = (
    all_features
    .merge(uo_features,      on='stay_id', how='left')
    .merge(vaso_features,    on='stay_id', how='left')
    .merge(vent_features,    on='stay_id', how='left')
    .merge(culture_features, on='stay_id', how='left')
)

# Median-fill the new columns
new_gap_cols = [
    'uo_total_24h', 'uo_mean_hourly', 'uo_min_hourly', 'uo_count', 'oliguria_flag',
    'vaso_events_24h', 'vasopressor_flag', 'ventilated_flag',
    'blood_culture_drawn', 'blood_culture_positive'
]
new_gap_cols_present = [c for c in new_gap_cols if c in all_features.columns]
gap_medians = all_features[new_gap_cols_present].median()
all_features[new_gap_cols_present] = all_features[new_gap_cols_present].fillna(gap_medians)

# Re-save
all_features.to_csv(OUTPUT_DIR / 'engineered_features.csv', index=False)

feature_cols_all = [c for c in all_features.columns if c != 'stay_id']
with open(OUTPUT_DIR / 'feature_names.txt', 'w') as f:
    f.write('\n'.join(feature_cols_all))

print(f'\nUpdated engineered_features.csv — shape: {all_features.shape}')
print(f'Total features (excl. stay_id): {len(feature_cols_all)}')
print(f'\nNew gap features added:')
for c in new_gap_cols_present:
    print(f'  {c}: {all_features[c].notna().sum():,} non-null')


## Summary of Labels

In [ ]:
print('\n--- Label Summary ---')
for horizon in ['6h', '12h', '24h']:
    y    = label_arrays[f'y_{horizon}']
    elig = label_arrays[f'eligible_{horizon}']
    print(f'Horizon {horizon:>3s} | eligible: {elig.sum():,} '
          f'| sepsis positive: {y[elig].sum():,} '
          f'| positive rate: {y[elig].mean():.3f}')


--- Label Summary ---
Horizon  6h | eligible: 64,815 | sepsis positive: 297 | positive rate: 0.005
Horizon 12h | eligible: 58,021 | sepsis positive: 453 | positive rate: 0.008
Horizon 24h | eligible: 46,157 | sepsis positive: 662 | positive rate: 0.014
